In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive monté")

Mounted at /content/drive
✓ Drive monté


In [2]:
import warnings
import json
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠ No GPU — Runtime → Change runtime type → GPU")

print("\n✓ Imports OK")

Device : cuda
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM   : 102.0 GB

✓ Imports OK


In [3]:
DRIVE_DIR = Path("/content/drive/MyDrive")
BASE_DIR  = Path("/content/aneo_work")

DATA_DIR   = DRIVE_DIR / "data_4x"
OUT_DIR    = BASE_DIR  / "aneo_output"
MODEL_DIR  = BASE_DIR  / "aneo_models"
DRIVE_MODEL_DIR = DATA_DIR / "models"  # persists across Colab disconnects
TENSOR_DIR = BASE_DIR  / "aneo_tensors"
FIG_DIR    = BASE_DIR  / "aneo_figures"

for d in [OUT_DIR, MODEL_DIR, TENSOR_DIR, FIG_DIR, DRIVE_MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ {d}")

print(f"\nAccès Drive source :")
print(f"  data_4x existe         : {DATA_DIR.exists()}")
print(f"  aneo_grid_points.csv   : {(DATA_DIR/'aneo_grid_points.csv').exists()}")
csv_files = list(DATA_DIR.glob("*.csv"))
print(f"  CSV files dans data_4x : {len(csv_files)}")
print(f"\n✓ Architecture deux disques OK")

✓ /content/aneo_work/aneo_output
✓ /content/aneo_work/aneo_models
✓ /content/aneo_work/aneo_tensors
✓ /content/aneo_work/aneo_figures

Accès Drive source :
  data_4x existe         : True
  aneo_grid_points.csv   : True
  CSV files dans data_4x : 32

✓ Architecture deux disques OK


In [4]:
if (TENSOR_DIR / "split.json").exists():
    print("✓ Tenseurs déjà créés — cellule ignorée")
    print(f"  Fichiers .npy : {len(list(TENSOR_DIR.glob('*.npy')))}")
else:
    print("Création des tenseurs depuis les CSV...")

    print("\nÉtape 1/4 — Chargement grid points...")
    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    print(f"  Grid points : {len(grid):,} points")

    NX, NY = 1000, 500
    assert len(grid) == NX * NY

    land_use_vals = sorted(grid['land_use_code'].dropna().unique())
    land_use_map  = {v: i for i, v in enumerate(land_use_vals)}
    grid['land_use_enc'] = grid['land_use_code'].map(
        land_use_map).fillna(0).astype(np.float32)
    lu_mean = float(grid['land_use_enc'].mean())
    lu_std  = float(grid['land_use_enc'].std()) + 1e-8
    grid['land_use_enc_norm'] = (grid['land_use_enc'] - lu_mean) / lu_std
    print(f"  land_use_map : {land_use_map}")

    print("\nÉtape 2/4 — Identification des fichiers...")
    obs_files = sorted(DATA_DIR.glob("aneo_observations_day_*.csv"))
    print(f"  Fichiers observations : {len(obs_files)}")

    OBS_COLS = [
        'point_id', 'day_index', 'hour_utc',
        'temperature_c', 'air_density_kg_m3',
        'relative_humidity_pct', 'pressure_pa',
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
    ]
    TARGET_COLS        = ['wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s']
    NUMERIC_INPUT_COLS = [
        'temperature_c', 'air_density_kg_m3',
        'relative_humidity_pct', 'pressure_pa',
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
    ]
    DAY_COL    = 'day_index'
    HOUR_COL   = 'hour_utc'
    N_INPUT_CH = len(NUMERIC_INPUT_COLS) + 1

    print("\nÉtape 3/4 — Calcul des stats de normalisation...")
    sum_dict  = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    sum2_dict = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    count     = 0
    days_set  = set()
    hours_set = set()

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        for col in NUMERIC_INPUT_COLS:
            vals = df[col].values.astype(np.float64)
            sum_dict[col]  += vals.sum()
            sum2_dict[col] += (vals**2).sum()
        days_set.update(df[DAY_COL].unique())
        hours_set.update(df[HOUR_COL].unique())
        count += len(df)
        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(obs_files)} fichiers analysés...")

    mean_dict = {c: sum_dict[c] / count for c in sum_dict}
    std_dict  = {
        c: float(np.sqrt(max(sum2_dict[c]/count - mean_dict[c]**2, 0) + 1e-8))
        for c in sum_dict
    }
    days  = sorted([int(d) for d in days_set])
    hours = sorted([int(h) for h in hours_set])

    print(f"  Jours  : {days}")
    print(f"  Heures : {hours}")
    print(f"  wind_u : mean={mean_dict['wind_u_m_s']:.4f}  std={std_dict['wind_u_m_s']:.4f}")
    print(f"  wind_v : mean={mean_dict['wind_v_m_s']:.4f}  std={std_dict['wind_v_m_s']:.4f}")
    print(f"  wind_w : mean={mean_dict['wind_w_m_s']:.4f}  std={std_dict['wind_w_m_s']:.4f}")

    norm_df = pd.DataFrame({
        col: {'mean': mean_dict[col], 'std': std_dict[col]}
        for col in TARGET_COLS
    }).T
    norm_df.to_csv(OUT_DIR / "normalization_stats.csv")

    print("\nÉtape 4/4 — Création des tenseurs .npy...")
    n_created = 0
    grid_indexed = grid.set_index('point_id')

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        df[DAY_COL]  = df[DAY_COL].astype(int)
        df[HOUR_COL] = df[HOUR_COL].astype(int)

        df = df.merge(
            grid_indexed[['ix', 'iy', 'land_use_enc_norm']],
            left_on='point_id', right_index=True, how='left')

        for col in NUMERIC_INPUT_COLS:
            df[col + '_norm'] = (
                (df[col] - mean_dict[col]) / std_dict[col]
            ).astype(np.float32)

        input_norm_cols  = [c + '_norm' for c in NUMERIC_INPUT_COLS] + ['land_use_enc_norm']
        target_norm_cols = [c + '_norm' for c in TARGET_COLS]

        for day in df[DAY_COL].unique():
            for hour in df[HOUR_COL].unique():
                snap = df[
                    (df[DAY_COL]==day) & (df[HOUR_COL]==hour)
                ].sort_values(['iy', 'ix'])

                x_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_X.npy"
                y_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_Y.npy"
                if x_path.exists():
                    n_created += 1
                    continue
                if len(snap) != NX * NY:
                    continue

                X = snap[input_norm_cols].values.reshape(NY, NX, N_INPUT_CH).astype(np.float32)
                Y = snap[target_norm_cols].values.reshape(NY, NX, len(TARGET_COLS)).astype(np.float32)
                np.save(x_path, X)
                np.save(y_path, Y)
                n_created += 1

        import gc; gc.collect()
        print(f"  [{i+1}/{len(obs_files)}] {f.name} — {n_created} snapshots créés")

    n_days     = len(days)
    val_days   = [days[i] for i in range(4, n_days, 5)]
    test_days  = [days[i] for i in range(2, n_days, 5)]
    train_days = [d for d in days if d not in val_days and d not in test_days]

    split = {
        "train_days": train_days, "val_days": val_days, "test_days": test_days,
        "heures": list(hours), "nx": NX, "ny": NY,
        "n_input": N_INPUT_CH, "n_target": len(TARGET_COLS),
        "input_cols": input_norm_cols, "target_cols": TARGET_COLS,
        "day_col": DAY_COL, "hour_col": HOUR_COL,
        "land_use_map": land_use_map, "lu_mean": lu_mean, "lu_std": lu_std,
        "mean_dict": mean_dict, "std_dict": std_dict,
    }
    with open(TENSOR_DIR / "split.json", "w") as f:
        json.dump(split, f, indent=2)

    print(f"\n  Train : {len(train_days)} jours → {train_days}")
    print(f"  Val   : {len(val_days)} jours → {val_days}")
    print(f"  Test  : {len(test_days)} jours → {test_days}")
    print(f"\n✓ Pipeline terminé → {TENSOR_DIR}")

Création des tenseurs depuis les CSV...

Étape 1/4 — Chargement grid points...
  Grid points : 500,000 points
  land_use_map : {'AGRICULTURE': 0, 'FOREST': 1, 'MOUNTAIN': 2, 'PLAIN': 3, 'URBAN_DENSE': 4, 'URBAN_MEDIUM': 5, 'WATER': 6}

Étape 2/4 — Identification des fichiers...
  Fichiers observations : 30

Étape 3/4 — Calcul des stats de normalisation...
  10/30 fichiers analysés...
  20/30 fichiers analysés...
  30/30 fichiers analysés...
  Jours  : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
  Heures : [0, 6, 12, 18]
  wind_u : mean=1.5706  std=2.0157
  wind_v : mean=4.1352  std=2.1649
  wind_w : mean=-0.0274  std=0.0624

Étape 4/4 — Création des tenseurs .npy...
  [1/30] aneo_observations_day_001_20260401_multitime.csv — 4 snapshots créés
  [2/30] aneo_observations_day_002_20260402_multitime.csv — 8 snapshots créés
  [3/30] aneo_observations_day_003_20260403_multitime.csv — 12 snapshots créés
  [4/30] aneo_observat

In [5]:
if (TENSOR_DIR / "grid_features.npy").exists() and (TENSOR_DIR / "terrain_raw.npy").exists():
    print("✓ grid_features.npy et terrain_raw.npy existent déjà")
else:
    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    NX_g, NY_g = 1000, 500

    def normalize(arr):
        return (arr - arr.mean()) / (arr.std() + 1e-8)

    alt_grid     = normalize(grid['z_terrain_m'].values).reshape(NY_g, NX_g)
    zrel_grid    = normalize(grid['z_rel_m'].values).reshape(NY_g, NX_g)
    z0_grid      = normalize(grid['z0_m'].values).reshape(NY_g, NX_g)
    slope_grid   = normalize(grid['slope_deg'].values).reshape(NY_g, NX_g)
    aspect_rad   = np.deg2rad(grid['aspect_deg'].values)
    asp_sin_grid = np.sin(aspect_rad).reshape(NY_g, NX_g).astype(np.float32)
    asp_cos_grid = np.cos(aspect_rad).reshape(NY_g, NX_g).astype(np.float32)
    x_grid       = normalize(grid['x_m'].values).reshape(NY_g, NX_g)
    y_grid       = normalize(grid['y_m'].values).reshape(NY_g, NX_g)

    grid_features = np.stack([
        alt_grid, zrel_grid, z0_grid, slope_grid,
        asp_sin_grid, asp_cos_grid, x_grid, y_grid,
    ], axis=-1).astype(np.float32)

    np.save(TENSOR_DIR / "grid_features.npy", grid_features)
    print(f"  grid_features shape : {grid_features.shape}")
    print(f"✓ grid_features.npy sauvegardé")

    # ── z_terrain_m et z_rel_m en unités physiques (mètres), pour la loss physique ──
    # CORRECTION : L_mass a besoin des valeurs réelles (pas z-scoreées) pour
    # calculer une vraie pente de terrain et une hauteur de capteur en mètres.
    z_terrain_raw = grid['z_terrain_m'].values.reshape(NY_g, NX_g).astype(np.float32)
    z_rel_raw     = grid['z_rel_m'].values.reshape(NY_g, NX_g).astype(np.float32)
    np.save(TENSOR_DIR / "terrain_raw.npy",
            np.stack([z_terrain_raw, z_rel_raw], axis=0))
    print(f"✓ terrain_raw.npy sauvegardé (z_terrain_m, z_rel_m physiques)")

  grid_features shape : (500, 1000, 8)
✓ grid_features.npy sauvegardé


In [6]:
with open(TENSOR_DIR / "split.json", "r") as f:
    split = json.load(f)

TRAIN_DAYS  = split["train_days"]
VAL_DAYS    = split["val_days"]
TEST_DAYS   = split["test_days"]
HEURES      = split["heures"]
NX          = split["nx"]
NY          = split["ny"]
N_INPUT     = split["n_input"]
N_TARGET    = split["n_target"]
INPUT_COLS  = split["input_cols"]
TARGET_COLS = split["target_cols"]

norm_stats = pd.read_csv(OUT_DIR / "normalization_stats.csv", index_col=0)

STD_U = norm_stats.loc["wind_u_m_s", "std"]
STD_V = norm_stats.loc["wind_v_m_s", "std"]
STD_W = norm_stats.loc["wind_w_m_s", "std"]
MU_U  = norm_stats.loc["wind_u_m_s", "mean"]
MU_V  = norm_stats.loc["wind_v_m_s", "mean"]
MU_W  = norm_stats.loc["wind_w_m_s", "mean"]

# ── CORRECTION 1 SETUP : stats densité pour dénormalisation dans L_mass ──────
MU_RHO  = float(norm_stats.loc["air_density_kg_m3", "mean"])
STD_RHO = float(norm_stats.loc["air_density_kg_m3", "std"])

N_GRID_FEATURES     = 8
N_TEMPORAL_FEATURES = 2
N_INPUT_EFFECTIVE   = N_INPUT + N_GRID_FEATURES + N_TEMPORAL_FEATURES  # 18

print("Configuration :\n")
print(f"  Train : {len(TRAIN_DAYS)} jours → {TRAIN_DAYS}")
print(f"  Val   : {len(VAL_DAYS)} jours → {VAL_DAYS}")
print(f"  Test  : {len(TEST_DAYS)} jours → {TEST_DAYS}")
print(f"\n  Grille            : {NX} × {NY}")
print(f"  N_INPUT_EFFECTIVE : {N_INPUT_EFFECTIVE}")
print(f"  N_TARGET          : {N_TARGET}")
print(f"\n  STD u,v,w : {STD_U:.4f}, {STD_V:.4f}, {STD_W:.4f}")
print(f"  MU  u,v,w : {MU_U:.4f}, {MU_V:.4f}, {MU_W:.4f}")
print(f"\n  ρ physique : μ={MU_RHO:.4f} kg/m³  σ={STD_RHO:.4f} kg/m³")
print(f"\n✓ Config OK")

KeyError: 'air_density_kg_m3'

In [ ]:
class ANEODataset(Dataset):
    """
    18 canaux: 8 originaux + 8 terrain + 2 temporels.
    Pas de rotation (terrain statique).
    """
    def __init__(self, days, heures, tensor_dir, augment=False):
        self.tensor_dir = Path(tensor_dir)
        self.augment    = augment
        self.samples    = []

        for day in days:
            for h in heures:
                x_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_X.npy"
                y_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_Y.npy"
                if x_path.exists() and y_path.exists():
                    self.samples.append((x_path, y_path, day, h))

        gf = np.load(self.tensor_dir / "grid_features.npy")
        self.grid_features = torch.from_numpy(
            gf.transpose(2, 0, 1).astype(np.float32))

        print(f"  Dataset : {len(self.samples)} snapshots "
              f"({'aug ON' if augment else 'aug OFF'})  "
              f"{N_INPUT_EFFECTIVE} channels")

    def __len__(self):
        return len(self.samples)

    def augment_sample(self, X, Y):
        X, Y = X.clone(), Y.clone()
        if torch.rand(1) > 0.5:
            scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
            X[4:7] *= scale
            Y      *= scale
        if torch.rand(1) > 0.5:
            X[:8] += torch.randn_like(X[:8]) * 0.02
        return X, Y

    def __getitem__(self, idx):
        x_path, y_path, day, h = self.samples[idx]

        X = torch.from_numpy(
            np.load(x_path).astype(np.float32).transpose(2, 0, 1))
        Y = torch.from_numpy(
            np.load(y_path).astype(np.float32).transpose(2, 0, 1))

        if self.augment:
            X, Y = self.augment_sample(X, Y)

        X = torch.cat([X, self.grid_features], dim=0)

        _, H, W = X.shape
        hour_sin = float(np.sin(2 * np.pi * h / 24))
        hour_cos = float(np.cos(2 * np.pi * h / 24))
        sin_ch = torch.full((1, H, W), hour_sin, dtype=torch.float32)
        cos_ch = torch.full((1, H, W), hour_cos, dtype=torch.float32)
        X = torch.cat([X, sin_ch, cos_ch], dim=0)

        return X, Y, day, h

print("✓ ANEODataset défini")

In [ ]:
BATCH_SIZE = 1

train_dataset = ANEODataset(
    TRAIN_DAYS + VAL_DAYS, HEURES, TENSOR_DIR, augment=True)
val_dataset   = ANEODataset(
    VAL_DAYS,   HEURES, TENSOR_DIR, augment=False)
test_dataset  = ANEODataset(
    TEST_DAYS,  HEURES, TENSOR_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1,
                          shuffle=False, num_workers=2)

print(f"Batch size     : {BATCH_SIZE}")
print(f"Train batches  : {len(train_loader)}")
print(f"Val   batches  : {len(val_loader)}")
print(f"Test  batches  : {len(test_loader)}")

X_b, Y_b, _, _ = next(iter(train_loader))
print(f"\nX : {X_b.shape}  ← doit être [1, 18, 500, 1000]")
print(f"Y : {Y_b.shape}  ← doit être [1,  3, 500, 1000]")

assert X_b.shape[1] == N_INPUT_EFFECTIVE, \
    f"Channels incorrects: {X_b.shape[1]} vs {N_INPUT_EFFECTIVE}"

print(f"\n✓ Dataloaders prêts")

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x, modes_y):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes_x      = modes_x
        self.modes_y      = modes_y
        scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        B = x.shape[0]
        x_ft = torch.fft.rfft2(x.float())
        out_ft = torch.zeros(
            B, self.out_channels,
            x.size(-2), x.size(-1)//2+1,
            dtype=torch.cfloat, device=x.device)
        out_ft[:,:,:self.modes_x,:self.modes_y] = \
            self.compl_mul2d(
                x_ft[:,:,:self.modes_x,:self.modes_y], self.weights1)
        out_ft[:,:,-self.modes_x:,:self.modes_y] = \
            self.compl_mul2d(
                x_ft[:,:,-self.modes_x:,:self.modes_y], self.weights2)
        return torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))


class FNOBlock(nn.Module):
    def __init__(self, channels, modes_x, modes_y, dropout=0.05):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_x, modes_y)
        self.local    = nn.Conv2d(channels, channels, kernel_size=1)
        self.norm     = nn.InstanceNorm2d(channels, affine=True)
        self.dropout  = nn.Dropout2d(p=dropout)

    def forward(self, x):
        return F.gelu(self.dropout(
            self.norm(self.spectral(x) + self.local(x))))


class FNO2D(nn.Module):
    def __init__(self, in_channels, out_channels,
                 hidden_channels=96, n_blocks=5,
                 modes_x=20, modes_y=20, dropout=0.05):
        super().__init__()
        self.lifting = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            FNOBlock(hidden_channels, modes_x, modes_y, dropout)
            for _ in range(n_blocks)
        ])
        self.skip_proj = nn.Conv2d(
            hidden_channels, hidden_channels, kernel_size=1)

        def make_head(h):
            return nn.Sequential(
                nn.Conv2d(h, h//2, kernel_size=1), nn.GELU(),
                nn.Conv2d(h//2, h//4, kernel_size=1), nn.GELU(),
                nn.Conv2d(h//4, 1,   kernel_size=1),
            )

        self.head_u = make_head(hidden_channels)
        self.head_v = make_head(hidden_channels)
        self.head_w = make_head(hidden_channels)

    def forward(self, x):
        x    = self.lifting(x)
        skip = self.skip_proj(x)
        for block in self.blocks:
            x = block(x)
        x = x + skip
        return torch.cat([
            self.head_u(x),
            self.head_v(x),
            self.head_w(x),
        ], dim=1)


model = FNO2D(
    in_channels     = N_INPUT_EFFECTIVE,
    out_channels    = N_TARGET,
    hidden_channels = 96,
    n_blocks        = 5,
    modes_x         = 20,
    modes_y         = 20,
    dropout         = 0.05,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FNO2D Model :")
print(f"  in_channels     : {N_INPUT_EFFECTIVE}")
print(f"  hidden_channels : 96")
print(f"  n_blocks        : 5")
print(f"  modes           : 20×20")
print(f"  paramètres      : {n_params:,}")
print(f"  device          : {device}")
print(f"\n✓ Modèle prêt")

In [ ]:
class PhysicsConstrainedLoss(nn.Module):
    def __init__(self, lambda_mass=0.3, lambda_bc=0.05,
                 lambda_smooth=0.01, lambda_ri=0.02, dx=50.0,
                 mu_rho=1.154, std_rho=0.021, terrain_path=None):
        super().__init__()
        self.lambda_mass   = lambda_mass
        self.lambda_bc     = lambda_bc
        self.lambda_smooth = lambda_smooth
        self.lambda_ri     = lambda_ri
        self.dx            = dx
        # ── ρ physique : stocker les stats de dénormalisation ──────────────────────
        self.register_buffer('mu_rho',  torch.tensor(mu_rho,  dtype=torch.float32))
        self.register_buffer('std_rho', torch.tensor(std_rho, dtype=torch.float32))

        # ── CORRECTION 2 : terrain physique pour ∂(ρw)/∂z et pente du sol ─────
        # z_terrain (altitude du sol) et z_rel (hauteur du capteur au-dessus du
        # sol) en mètres, statiques sur tout le domaine (pas de rotation/crop,
        # même grille pour chaque échantillon — voir ANEODataset).
        terrain = np.load(terrain_path)  # (2, H, W) = [z_terrain_m, z_rel_m]
        self.register_buffer(
            'z_terrain', torch.from_numpy(terrain[0]).float()[None, None])
        self.register_buffer(
            'z_rel', torch.from_numpy(terrain[1]).float()[None, None])

    def forward(self, pred, target, rho_norm, phys_weight=1.0):
        # ── Loss données ──────────────────────────────────
        L_data = F.mse_loss(pred, target)

        # ── CORRECTION 1 : dénormaliser ρ avant de calculer les flux de masse ─
        # rho_norm est normalisée (∈ [-2,2])
        # rho_phys est en kg/m³ réels (∈ [1.06, 1.21])
        rho_phys = rho_norm * self.std_rho + self.mu_rho

        u_pred = pred[:, 0:1, :, :]
        v_pred = pred[:, 1:2, :, :]
        w_pred = pred[:, 2:3, :, :]

        # Flux de masse avec ρ physique
        rho_u = rho_phys * u_pred
        rho_v = rho_phys * v_pred

        # ── CORRECTION 2a : pas horizontal corrigé par la pente du terrain ──
        # Un pas de dx sur la grille traverse aussi une dénivelée dz_terrain
        # (le terrain tunisien n'est pas plat : sigma(z_terrain) ~ 109m sur data_4x
        # même à 50m d'espacement) ; la distance réellement parcourue est
        # sqrt(dx^2 + dz^2), pas dx tout court.
        dzt_dx = self.z_terrain[:, :, :, 2:] - self.z_terrain[:, :, :, :-2]
        dzt_dy = self.z_terrain[:, :, 2:, :] - self.z_terrain[:, :, :-2, :]
        ds_x = torch.sqrt((2 * self.dx) ** 2 + dzt_dx ** 2)
        ds_y = torch.sqrt((2 * self.dx) ** 2 + dzt_dy ** 2)

        # Divergence horizontale — différences finies centrées, pas corrigé
        drhou_dx = (rho_u[:, :, :, 2:] - rho_u[:, :, :, :-2]) / ds_x
        drhov_dy = (rho_v[:, :, 2:, :] - rho_v[:, :, :-2, :]) / ds_y
        min_y    = min(drhou_dx.shape[2], drhov_dy.shape[2])
        min_x    = min(drhou_dx.shape[3], drhov_dy.shape[3])

        # ── CORRECTION 2b : terme vertical ∂(ρw)/∂z, absent jusqu'ici ───
        # Le modèle ne prédit qu'une seule couche, à hauteur capteur z_rel (pas
        # de vérité-terrain à d'autres altitudes dans ANEO). On utilise la
        # condition limite w=0 au sol (z=0) comme second point pour une
        # différence finie sur la colonne [0, z_rel] : c'est une approximation
        # de premier ordre, pas une vraie dérivée verticale multi-niveaux.
        rho_w    = rho_phys * w_pred
        drhow_dz = rho_w / (self.z_rel + 1e-3)

        div = (drhou_dx[:, :, :min_y, :min_x] +
               drhov_dy[:, :, :min_y, :min_x] +
               drhow_dz[:, :, :min_y, :min_x])
        L_mass = (div ** 2).mean()

        # ── Condition aux limites : w = 0 au sol ──────────────
        L_bc = (w_pred[:, :, 0, :] ** 2).mean()

        # ── Continuité spatiale ───────────────────────
        grad_u_x = pred[:, 0:1, :, 1:] - pred[:, 0:1, :, :-1]
        grad_u_y = pred[:, 0:1, 1:, :] - pred[:, 0:1, :-1, :]
        grad_v_x = pred[:, 1:2, :, 1:] - pred[:, 1:2, :, :-1]
        grad_v_y = pred[:, 1:2, 1:, :] - pred[:, 1:2, :-1, :]
        L_smooth = ((grad_u_x ** 2).mean() + (grad_u_y ** 2).mean() +
                    (grad_v_x ** 2).mean() + (grad_v_y ** 2).mean())

        # ── Stabilité de Richardson ────────────────────
        du_dx = (u_pred[:, :, :, 1:] - u_pred[:, :, :, :-1]) / self.dx
        dv_dy = (v_pred[:, :, 1:, :] - v_pred[:, :, :-1, :]) / self.dx
        L_ri  = (F.relu(du_dx ** 2 - 1.0).mean() +
                 F.relu(dv_dy ** 2 - 1.0).mean())

        phys_term = (self.lambda_mass   * L_mass +
                     self.lambda_bc     * L_bc +
                     self.lambda_smooth * L_smooth +
                     self.lambda_ri     * L_ri)

        L_total = L_data + phys_weight * phys_term

        # ── Retourne 6 valeurs pour le logging des composantes ─────────
        return L_total, L_data, L_mass, L_bc, L_smooth, L_ri


criterion = PhysicsConstrainedLoss(
    lambda_mass=0.3, lambda_bc=0.05,
    lambda_smooth=0.01, lambda_ri=0.02, dx=50.0,
    mu_rho=MU_RHO, std_rho=STD_RHO,   # ← stats depuis normalization_stats.csv
    terrain_path=TENSOR_DIR / "terrain_raw.npy",
).to(device)
print(f"✓ Physics loss corrigée (∂(ρw)/∂z + pente terrain ajoutés)")
print(f"  ρ physique : μ={MU_RHO:.4f} kg/m³  σ={STD_RHO:.4f} kg/m³")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 3e-4,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
)

N_EPOCHS      = 150
WARMUP_EPOCHS = 15
EARLY_STOP    = 30
ACCUM_STEPS   = 4

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6)

scaler = GradScaler('cuda')

print(f"  N_EPOCHS      : {N_EPOCHS}")
print(f"  WARMUP_EPOCHS : {WARMUP_EPOCHS}")
print(f"  EARLY_STOP    : {EARLY_STOP}")
print(f"  ACCUM_STEPS   : {ACCUM_STEPS}")
print(f"\n✓ Optimizer prêt")

In [ ]:
model.train()
X_b, Y_b, _, _ = next(iter(train_loader))
X_b, Y_b = X_b.to(device), Y_b.to(device)
rho_norm = X_b[:, 1:2, :, :]

with autocast('cuda', dtype=torch.float16):
    pred = model(X_b)
    loss, l_data, l_mass, l_bc, l_smooth, l_ri = criterion(
        pred, Y_b, rho_norm, phys_weight=0.0)

assert pred.shape == Y_b.shape, \
    f"Shape mismatch: pred={pred.shape} vs {Y_b.shape}"

scaler.scale(loss).backward()
optimizer.zero_grad()

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"pred shape    : {pred.shape} ✓")
print(f"L_data        : {l_data.item():.6f}")
print(f"L_mass        : {l_mass.item():.6f}")
print(f"L_bc          : {l_bc.item():.6f}")
print(f"Paramètres    : {sum(p.numel() for p in model.parameters()):,}")
print(f"GPU total     : {total:.1f} GB")
print(f"GPU alloué    : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU réservé   : {torch.cuda.memory_reserved()/1e9:.2f} GB")
torch.cuda.empty_cache()
print(f"\n✓ OOM check OK")

In [ ]:
def physics_warmup(epoch, warmup_epochs):
    if warmup_epochs <= 0:
        return 1.0
    return min(1.0, (epoch + 1) / warmup_epochs)


def compute_r2_per_channel(pred, true):
    pred_np = pred.detach().float().cpu().numpy()
    true_np = true.detach().float().cpu().numpy()
    scores  = []
    for c in range(pred_np.shape[1]):
        y_true = true_np[:, c].flatten()
        y_pred = pred_np[:, c].flatten()
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        scores.append(1 - ss_res / (ss_tot + 1e-8))
    return np.mean(scores), scores


def train_epoch(epoch):
    model.train()
    train_loss = 0.0
    sum_data = sum_mass = sum_bc = sum_smooth = sum_ri = 0.0
    phys_w   = physics_warmup(epoch, WARMUP_EPOCHS)
    optimizer.zero_grad()

    for step, (X_b, Y_b, _, _) in enumerate(train_loader):
        X_b, Y_b  = X_b.to(device), Y_b.to(device)
        rho_norm  = X_b[:, 1:2, :, :]

        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b)
            loss, l_data, l_mass, l_bc, l_smooth, l_ri = criterion(
                pred, Y_b, rho_norm, phys_weight=phys_w)
            loss_scaled = loss / ACCUM_STEPS

        # ── CORRECTION 2 : GradScaler correctement utilisé ───────────────────
        scaler.scale(loss_scaled).backward()

        if (step+1) % ACCUM_STEPS == 0 or (step+1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item()
        sum_data   += l_data.item()
        sum_mass   += l_mass.item()
        sum_bc     += l_bc.item()
        sum_smooth += l_smooth.item()
        sum_ri     += l_ri.item()

    n = len(train_loader)
    return (train_loss/n,
            sum_data/n, sum_mass/n, sum_bc/n, sum_smooth/n, sum_ri/n)


def validate_epoch(epoch):
    model.eval()
    val_loss      = 0.0
    r2_means      = []
    r2_per_ch_sum = np.zeros(3)
    phys_w        = physics_warmup(epoch, WARMUP_EPOCHS)
    sum_data = sum_mass = sum_bc = 0.0

    with torch.no_grad():
        for X_b, Y_b, _, _ in val_loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            rho_norm = X_b[:, 1:2, :, :]

            # Pas d'autocast en validation → stabilité numérique maximale
            pred = model(X_b.float())
            loss, l_data, l_mass, l_bc, _, _ = criterion(
                pred, Y_b, rho_norm, phys_weight=phys_w)

            val_loss  += loss.item()
            sum_data  += l_data.item()
            sum_mass  += l_mass.item()
            sum_bc    += l_bc.item()

            r2_mean, r2_ch = compute_r2_per_channel(pred, Y_b)
            r2_means.append(r2_mean)
            r2_per_ch_sum += np.array(r2_ch)

    n = len(val_loader)
    return (val_loss/n,
            np.mean(r2_means),
            r2_per_ch_sum/n,
            sum_data/n, sum_mass/n, sum_bc/n)


print("✓ Fonctions définies (GradScaler corrigé + logging composantes)")

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'r2': [], 'r2_u': [], 'r2_v': [], 'r2_w': [],
    'lr': [], 'phys_weight': [],
    'train_l_data': [], 'train_l_mass': [], 'train_l_bc': [],
    'train_l_smooth': [], 'train_l_ri': [],
    'val_l_data': [], 'val_l_mass': [], 'val_l_bc': [],
}

best_val_loss    = float('inf')
best_epoch       = 0
best_r2          = -np.inf
patience_counter = 0

print("="*108)
print("STARTING FNO2D TRAINING — CORRECTIONS APPLIQUÉES")
print("  ✓ ρ physique dans L_mass  ✓ GradScaler actif  ✓ Logging composantes")
print("="*108)
print(f"{'Ep':>4} | {'Train':>8} | {'Val':>8} | {'R²avg':>6} | "
      f"{'R²_u':>6} | {'R²_v':>6} | {'R²_w':>6} | "
      f"{'L_data':>7} | {'L_mass':>7} | {'ratio':>7} | "
      f"{'φ_w':>5} | {'LR':>8} | Status")
print("-"*108)

try:
    for epoch in range(N_EPOCHS):
        (tr_loss, tr_data, tr_mass,
         tr_bc, tr_smooth, tr_ri)          = train_epoch(epoch)
        (va_loss, r2, r2_ch,
         va_data, va_mass, va_bc)          = validate_epoch(epoch)
        scheduler.step()

        lr        = optimizer.param_groups[0]['lr']
        phys_w    = physics_warmup(epoch, WARMUP_EPOCHS)
        # Ratio clé : si < 0.001 après warmup → physique ignorée
        mass_ratio = (0.3 * tr_mass) / (tr_data + 1e-8)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['r2'].append(r2)
        history['r2_u'].append(r2_ch[0])
        history['r2_v'].append(r2_ch[1])
        history['r2_w'].append(r2_ch[2])
        history['lr'].append(lr)
        history['phys_weight'].append(phys_w)
        history['train_l_data'].append(tr_data)
        history['train_l_mass'].append(tr_mass)
        history['train_l_bc'].append(tr_bc)
        history['train_l_smooth'].append(tr_smooth)
        history['train_l_ri'].append(tr_ri)
        history['val_l_data'].append(va_data)
        history['val_l_mass'].append(va_mass)
        history['val_l_bc'].append(va_bc)

        print(f"{epoch+1:>4} | {tr_loss:>8.5f} | {va_loss:>8.5f} | "
              f"{r2:>6.4f} | {r2_ch[0]:>6.4f} | {r2_ch[1]:>6.4f} | "
              f"{r2_ch[2]:>6.4f} | {tr_data:>7.4f} | {tr_mass:>7.4f} | "
              f"{mass_ratio:>7.4f} | {phys_w:>5.2f} | {lr:>8.2e} | ", end="")

        if phys_w >= 1.0 and mass_ratio < 0.001:
            print("⚠ PHYS FAIBLE ", end="")

        if va_loss < best_val_loss:
            best_val_loss    = va_loss
            best_epoch       = epoch + 1
            best_r2          = r2
            patience_counter = 0
            torch.save({
                'epoch'               : epoch + 1,
                'model_state_dict'    : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict'   : scaler.state_dict(),
                'val_loss'            : va_loss,
                'r2'                  : r2,
                'r2_u'                : r2_ch[0],
                'r2_v'                : r2_ch[1],
                'r2_w'                : r2_ch[2],
                'history'             : history,
                'val_l_data'          : va_data,
                'val_l_mass'          : va_mass,
                'val_l_bc'            : va_bc,
            }, MODEL_DIR / 'best_fno2d_model.pth')
            shutil.copy2(MODEL_DIR / 'best_fno2d_model.pth',
                         DRIVE_MODEL_DIR / 'best_fno2d_model.pth')
            print("✓ BEST")
        else:
            patience_counter += 1
            print(f"(patience {patience_counter}/{EARLY_STOP})"
                  if patience_counter % 5 == 0 else "")

        if patience_counter >= EARLY_STOP:
            print(f"\n⚠ Early stopping à l'époque {epoch+1}")
            break

except KeyboardInterrupt:
    print("\n⚠ Entraînement interrompu")
except Exception as e:
    print(f"\n✗ Échec: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

print("\n" + "="*108)
print("ENTRAÎNEMENT TERMINÉ")
print(f"Meilleure époque : {best_epoch}  "
      f"val_loss : {best_val_loss:.6f}  "
      f"R² : {best_r2:.4f}")
print("="*108)

pd.DataFrame(history).to_csv(
    OUT_DIR / 'fno2d_training_history.csv', index=False)
print(f"✓ Historique complet sauvegardé")
shutil.copy2(MODEL_DIR / 'best_fno2d_model.pth',
             DRIVE_MODEL_DIR / 'best_fno2d_model.pth')
shutil.copy2(OUT_DIR / 'fno2d_training_history.csv',
             DRIVE_MODEL_DIR / 'fno2d_training_history.csv')
print(f"✓ Checkpoint + historique copiés vers Drive ({DRIVE_MODEL_DIR})")

In [ ]:
ckpt = torch.load(MODEL_DIR / "best_fno2d_model.pth",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Chargé époque {ckpt['epoch']}  "
      f"val_loss={ckpt['val_loss']:.6f}  "
      f"R²avg={ckpt['r2']:.4f}")

results = []
with torch.no_grad():
    for X_b, Y_b, day, hour in test_loader:
        day, hour = day.item(), hour.item()
        pred  = model(X_b.to(device).float())
        pred_n = pred[0].float().cpu().numpy()
        true_n = Y_b[0].numpy()

        u_p = pred_n[0]*STD_U+MU_U; v_p = pred_n[1]*STD_V+MU_V
        w_p = pred_n[2]*STD_W+MU_W
        u_t = true_n[0]*STD_U+MU_U; v_t = true_n[1]*STD_V+MU_V
        w_t = true_n[2]*STD_W+MU_W

        mae_u  = np.mean(np.abs(u_p-u_t))
        mae_v  = np.mean(np.abs(v_p-v_t))
        mae_w  = np.mean(np.abs(w_p-w_t))
        rmse_u = np.sqrt(np.mean((u_p-u_t)**2))
        rmse_v = np.sqrt(np.mean((v_p-v_t)**2))
        rmse_w = np.sqrt(np.mean((w_p-w_t)**2))
        r2_u   = 1-np.sum((u_p-u_t)**2)/(np.sum((u_t-u_t.mean())**2)+1e-9)
        r2_v   = 1-np.sum((v_p-v_t)**2)/(np.sum((v_t-v_t.mean())**2)+1e-9)
        r2_w   = 1-np.sum((w_p-w_t)**2)/(np.sum((w_t-w_t.mean())**2)+1e-9)
        bias_u = np.mean(u_p-u_t)
        bias_v = np.mean(v_p-v_t)
        bias_w = np.mean(w_p-w_t)
        ws_p   = np.sqrt(u_p**2+v_p**2); ws_t = np.sqrt(u_t**2+v_t**2)
        mae_ws = np.mean(np.abs(ws_p-ws_t))
        dir_p  = np.degrees(np.arctan2(v_p,u_p))%360
        dir_t  = np.degrees(np.arctan2(v_t,u_t))%360
        dir_err= np.minimum(np.abs(dir_p-dir_t), 360-np.abs(dir_p-dir_t))
        mae_dir= np.mean(dir_err)
        acc_u_05 = (np.abs(u_p-u_t)<0.5).mean()*100
        acc_u_10 = (np.abs(u_p-u_t)<1.0).mean()*100
        acc_v_05 = (np.abs(v_p-v_t)<0.5).mean()*100
        acc_v_10 = (np.abs(v_p-v_t)<1.0).mean()*100
        acc_w_01 = (np.abs(w_p-w_t)<0.1).mean()*100

        results.append(dict(
            day=day, hour=hour,
            mae_u=mae_u, mae_v=mae_v, mae_w=mae_w,
            rmse_u=rmse_u, rmse_v=rmse_v, rmse_w=rmse_w,
            r2_u=r2_u, r2_v=r2_v, r2_w=r2_w,
            bias_u=bias_u, bias_v=bias_v, bias_w=bias_w,
            mae_ws=mae_ws, mae_dir=mae_dir,
            acc_u_05=acc_u_05, acc_u_10=acc_u_10,
            acc_v_05=acc_v_05, acc_v_10=acc_v_10, acc_w_01=acc_w_01,
        ))
        print(f"  J{day:3d} h{hour:02d} | "
              f"MAE u={mae_u:.4f} v={mae_v:.4f} w={mae_w:.4f} | "
              f"R² u={r2_u:.4f} v={r2_v:.4f} w={r2_w:.4f} | "
              f"Acc_u={acc_u_05:.1f}% Acc_v={acc_v_05:.1f}%")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'fno2d_evaluation_results.csv', index=False)

print(f"\n{'='*72}")
print(f"TEST SET METRICS (moyenne sur {len(df_r)} snapshots):")
print(f"{'='*72}")
for col in ['mae_u','mae_v','mae_w','rmse_u','rmse_v','rmse_w',
            'r2_u','r2_v','r2_w','bias_u','bias_v','bias_w',
            'mae_ws','mae_dir','acc_u_05','acc_u_10',
            'acc_v_05','acc_v_10','acc_w_01']:
    print(f"  {col:12s}: {df_r[col].mean():.4f}")
print('='*72)
print(f"\n✓ Résultats sauvegardés")

In [ ]:
# Visualiser les composantes de loss pour vérifier que la physique est active
def plot_loss_components(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ep = range(1, len(history['train_loss'])+1)

    axes[0].plot(ep, history['train_loss'], label='Train total')
    axes[0].plot(ep, history['val_loss'],   label='Val total')
    axes[0].set_title('Loss totale'); axes[0].legend()
    axes[0].set_yscale('log'); axes[0].set_xlabel('Époque')

    axes[1].plot(ep, history['train_l_data'],   label='L_data',   lw=2)
    axes[1].plot(ep, history['train_l_mass'],   label='L_mass',   lw=2)
    axes[1].plot(ep, history['train_l_bc'],     label='L_bc',     lw=1.5)
    axes[1].plot(ep, history['train_l_smooth'], label='L_smooth', lw=1.5)
    axes[1].plot(ep, history['train_l_ri'],     label='L_ri',     lw=1.5)
    axes[1].set_title('Composantes loss (train)')
    axes[1].legend(fontsize=8); axes[1].set_yscale('log')
    axes[1].set_xlabel('Époque')

    # Ratio critique : λ_mass·L_mass / L_data
    ratio = [0.3*m/(d+1e-8)
             for m,d in zip(history['train_l_mass'], history['train_l_data'])]
    axes[2].plot(ep, ratio, color='red', lw=2)
    axes[2].axhline(0.01, color='orange', ls='--', label='Seuil minimal 1%')
    axes[2].axhline(0.1,  color='green',  ls='--', label='Seuil correct 10%')
    axes[2].set_title('Ratio λ·L_mass / L_data\n(>0.01 = physique active)')
    axes[2].set_xlabel('Époque'); axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'loss_components.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Sauvegardé → aneo_output/loss_components.png")

plot_loss_components(history)

In [ ]:
# ── Visualisation 3D du champ de vent reconstruit (PyVista) ──────────────────
!pip install -q pyvista
import pyvista as pv
pv.start_xvfb()
pv.set_jupyter_backend('static')

STEP_SURF   = 5    # sous-echantillonnage de la surface terrain (1000x500 -> 200x100)
STEP_ARROWS = 25   # sous-echantillonnage des fleches de vent

grid_raw  = pd.read_csv(DATA_DIR / "aneo_grid_points.csv").sort_values(['iy', 'ix']).reset_index(drop=True)
X_terrain = grid_raw['x_m'].values.reshape(NY, NX)
Y_terrain = grid_raw['y_m'].values.reshape(NY, NX)
Z_terrain = grid_raw['z_terrain_m'].values.reshape(NY, NX)

X_b, Y_b, day, hour = next(iter(test_loader))
with torch.no_grad():
    pred = model(X_b.to(device).float())[0].float().cpu().numpy()
day, hour = day.item(), hour.item()

u_p   = pred[0] * STD_U + MU_U
v_p   = pred[1] * STD_V + MU_V
w_p   = pred[2] * STD_W + MU_W
speed = np.sqrt(u_p**2 + v_p**2 + w_p**2)

Xs = X_terrain[::STEP_SURF, ::STEP_SURF]
Ys = Y_terrain[::STEP_SURF, ::STEP_SURF]
Zs = Z_terrain[::STEP_SURF, ::STEP_SURF]
Ss = speed[::STEP_SURF, ::STEP_SURF]

terrain = pv.StructuredGrid(Xs, Ys, Zs)
terrain["wind_speed_m_s"] = Ss.ravel(order="F")

Xa = X_terrain[::STEP_ARROWS, ::STEP_ARROWS]
Ya = Y_terrain[::STEP_ARROWS, ::STEP_ARROWS]
Za = Z_terrain[::STEP_ARROWS, ::STEP_ARROWS] + 15  # decolle du sol pour la lisibilite
Ua = u_p[::STEP_ARROWS, ::STEP_ARROWS]
Va = v_p[::STEP_ARROWS, ::STEP_ARROWS]
Wa = w_p[::STEP_ARROWS, ::STEP_ARROWS]

points  = np.column_stack([Xa.ravel(), Ya.ravel(), Za.ravel()])
vectors = np.column_stack([Ua.ravel(), Va.ravel(), Wa.ravel()])
cloud   = pv.PolyData(points)
cloud["vectors"] = vectors
cloud["speed"]   = np.linalg.norm(vectors, axis=1)
arrows = cloud.glyph(orient="vectors", scale="speed", factor=15)

plotter = pv.Plotter(window_size=(1100, 800))
plotter.add_mesh(terrain, scalars="wind_speed_m_s", cmap="viridis",
                  smooth_shading=True, show_scalar_bar=True)
plotter.add_mesh(arrows, color="white")
plotter.camera_position = "iso"
plotter.add_title(f"Champ de vent reconstruit -- jour {day} h{hour:02d}")

out_png = FIG_DIR / f"wind3d_day{day:03d}_h{hour:02d}.png"
plotter.show(screenshot=str(out_png))
print(f"Visualisation 3D sauvegardee -> {out_png}")


In [ ]:
import time, shutil
from google.colab import files

def periodic_zip_download(base_dir=BASE_DIR, interval=300):
    iteration = 0
    print(f"Backup démarré — toutes les {interval//60} minutes")
    while True:
        time.sleep(interval)
        iteration += 1
        zip_path = f"/content/aneo_backup_{iteration}"
        print(f"[{iteration}] Création du zip...", end=" ")
        shutil.make_archive(zip_path, 'zip', base_dir)
        size_mb = Path(zip_path + ".zip").stat().st_size / 1e6
        print(f"{size_mb:.0f} MB — téléchargement...")
        files.download(zip_path + ".zip")
        print(f"✓ aneo_backup_{iteration}.zip téléchargé")
        Path(zip_path + ".zip").unlink()

periodic_zip_download(base_dir=BASE_DIR, interval=300)